In [61]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,SampleID,ImagePath,Label
0,250,dataset/images/sample_0250.png,3
1,434,dataset/images/sample_0434.png,4
2,20,dataset/images/sample_0020.png,4
3,323,dataset/images/sample_0323.png,4
4,333,dataset/images/sample_0333.png,3


In [62]:
df_train.Label.value_counts()

Label
4    101
3     96
5     87
6     75
2     25
7     13
8      3
Name: count, dtype: int64

In [63]:
import torch

num_epochs = 10
batch_size = 32
learning_rate = 0.0001

device = torch.device("cuda")
device

device(type='cuda')

In [64]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T

train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ToTensor(),
])

valid_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

class MyDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        path = self.df.ImagePath.iloc[idx]
        img = Image.open(path)
        img = self.transform(img)
        label = torch.tensor(self.df.Label.iloc[idx], dtype=torch.float32).view(1)
        return img, label

from sklearn.model_selection import train_test_split
df_1, df_2 = train_test_split(df_train, test_size=0.2, random_state=42)

train_dataset = MyDataset(df_1, train_transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

valid_dataset = MyDataset(df_2, valid_transform)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size)

train_dataset[0][0][0][0][0]

tensor(0.)

In [65]:
import torch.nn as nn
import torchvision.models as models

model = models.resnet18(weights='DEFAULT')
model.fc = nn.Sequential(
    nn.Linear(512, 1)
)

model = model.to(device)
criterion = nn.MSELoss()
optim = torch.optim.AdamW(model.parameters(), lr=learning_rate)

model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [66]:
def train(model, epoch):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optim.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optim.step()
        total_loss += loss.item()
    print("Epoch: ", epoch)
    print("Training loss: ", total_loss/len(inputs))

def test(model, epoch):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for inputs, labels in valid_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
    print("Validation loss: ", total_loss/len(inputs))

In [67]:
for epoch in range(1, num_epochs+1):
    train(model, epoch)
    test(model, epoch)

Epoch:  1
Training loss:  5.836121082305908
Validation loss:  4.164940595626831
Epoch:  2
Training loss:  4.212732583284378
Validation loss:  2.844631314277649
Epoch:  3
Training loss:  2.5407475233078003
Validation loss:  2.0340983867645264
Epoch:  4
Training loss:  0.9660260751843452
Validation loss:  0.5755843073129654
Epoch:  5
Training loss:  0.2980347350239754
Validation loss:  0.20866761729121208
Epoch:  6
Training loss:  0.22713832464069128
Validation loss:  0.1486123912036419
Epoch:  7
Training loss:  0.15682382136583328
Validation loss:  0.13724958896636963
Epoch:  8
Training loss:  0.1479377755895257
Validation loss:  0.1589849852025509
Epoch:  9
Training loss:  0.14310528617352247
Validation loss:  0.13784102723002434
Epoch:  10
Training loss:  0.1196232633665204
Validation loss:  0.13812199607491493


In [68]:
final_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

class FinalDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        path = self.df.ImagePath.iloc[idx]
        img = Image.open(path)
        img = self.transform(img)
        return img

final_dataset = FinalDataset(df_test, final_transform)
final_loader = DataLoader(final_dataset, batch_size=batch_size)

final_dataset[0][0][0][0]

tensor(0.3882)

In [69]:
import numpy as np
model.eval()
preds = []

with torch.no_grad():
    for inputs in final_loader:
        inputs = inputs.to(device)
        outputs = model(inputs).squeeze(1).cpu().numpy()
        outputs = np.round(outputs)
        outputs = np.clip(outputs, 2, 8)
        preds.extend(outputs)

preds[:10]

[4.0, 2.0, 4.0, 6.0, 5.0, 3.0, 5.0, 5.0, 6.0, 2.0]

In [70]:
df_preds = pd.DataFrame({
    "SampleID": df_test.SampleID,
    "PredictedLabel": preds
})

df_preds

,SampleID,PredictedLabel
0,362,4.0
1,74,2.0
2,375,4.0
3,156,6.0
4,105,5.0
...,...,...
95,348,4.0
96,87,4.0
97,76,3.0
98,439,5.0


In [71]:
df_preds.PredictedLabel.value_counts()

PredictedLabel
4.0    28
5.0    28
3.0    24
6.0    14
2.0     6
Name: count, dtype: int64

In [72]:
df_preds.to_csv("submission.csv", index=False)